# Blood Report Analyzer - Real Excel Dataset

This notebook uses `../data/blood_report_dataset.xlsx` instead of sample data. It keeps the modelling trail visible: cleaning, imputations, EDA, visualisation, repeated tuning trials, a neural-network-style MLP trial, an optional Keras trial, and final joblib export for the Flask backend.

The deployed app collects nine CBC values, so the saved model remains compatible with those fields. The EDA still studies the wider workbook.

In [ ]:
import os, sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import GridSearchCV, RepeatedStratifiedKFold, StratifiedKFold, cross_validate, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.svm import SVC

sys.path.insert(0, str(Path('..', 'scripts').resolve()))
from train_blood import COL_MAP, FEATURES, TARGET, clean_blood_data

sns.set_theme(style='whitegrid', palette='Set2')
%matplotlib inline

ROOT = Path('..').resolve()
DATA_PATH = ROOT / 'data' / 'blood_report_dataset.xlsx'
MODELS_DIR = ROOT / 'models'
MODELS_DIR.mkdir(exist_ok=True)

## 1. Load the real Excel workbook

In [ ]:
df_raw = pd.read_excel(DATA_PATH, sheet_name='BloodReports')
print('Raw shape:', df_raw.shape)
display(df_raw.head())
missing = pd.DataFrame({
    'dtype': df_raw.dtypes.astype(str),
    'missing': df_raw.isna().sum(),
    'missing_pct': (df_raw.isna().mean() * 100).round(2),
}).sort_values('missing_pct', ascending=False)
display(missing.head(15))

## 2. Clean and prepare

The imported cleaner trims text categories, removes duplicates, fixes obvious Hb/MCV unit-entry issues, marks implausible clinical values missing, and keeps median imputation inside the model pipeline to avoid train/test leakage.

In [ ]:
df = clean_blood_data(df_raw)
print('Clean shape:', df.shape)
display(df[FEATURES + [TARGET]].head())
display(df[FEATURES].isna().sum().sort_values(ascending=False).to_frame('missing_before_pipeline_imputation'))

## 3. EDA and visualisations

In [ ]:
class_counts = df[TARGET].value_counts()
display(class_counts.to_frame('rows'))
plt.figure(figsize=(9, 5))
sns.barplot(x=class_counts.values, y=class_counts.index, color='#4C78A8')
plt.title('Blood diagnosis class balance')
plt.xlabel('Rows')
plt.ylabel('Diagnosis')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 11))
for ax, col in zip(axes.flat, FEATURES):
    sns.boxplot(data=df, x=TARGET, y=col, ax=ax, fliersize=1.5)
    ax.tick_params(axis='x', rotation=70)
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(df[FEATURES].corr(), annot=True, fmt='.2f', cmap='vlag', center=0)
plt.title('CBC feature correlations')
plt.tight_layout()
plt.show()

extra_cols = [c for c in ['age','height_cm','weight_kg','hematocrit','rdw','monocytes_pct','eosinophils_pct','basophils_pct'] if c in df]
display(df.groupby(TARGET)[FEATURES + extra_cols].median(numeric_only=True).round(2))

## 4. Split data and define trial helpers

In [ ]:
X = df[FEATURES]
y = df[TARGET].astype(str)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print('Train/test:', X_train.shape, X_test.shape)

def make_preprocessor():
    return ColumnTransformer([
        ('cbc', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', RobustScaler())]), FEATURES)
    ], remainder='drop', verbose_feature_names_out=False)

def evaluate_model(name, estimator):
    cv = RepeatedStratifiedKFold(n_splits=4, n_repeats=2, random_state=42)
    scores = cross_validate(estimator, X_train, y_train, cv=cv, scoring={'accuracy':'accuracy', 'f1_macro':'f1_macro'}, n_jobs=-1)
    estimator.fit(X_train, y_train)
    pred = estimator.predict(X_test)
    row = {
        'model': name,
        'cv_accuracy': scores['test_accuracy'].mean(),
        'cv_f1_macro': scores['test_f1_macro'].mean(),
        'test_accuracy': accuracy_score(y_test, pred),
        'test_f1_macro': f1_score(y_test, pred, average='macro'),
        'estimator': estimator,
    }
    print(f"{name:28s} cv_acc={row['cv_accuracy']:.3f} cv_f1={row['cv_f1_macro']:.3f} test_acc={row['test_accuracy']:.3f} test_f1={row['test_f1_macro']:.3f}")
    return row

## 5. Baseline models including a neural-network-style MLP

In [ ]:
baselines = {
    'random_forest_baseline': Pipeline([('prep', make_preprocessor()), ('model', RandomForestClassifier(n_estimators=350, min_samples_leaf=2, class_weight='balanced_subsample', random_state=42, n_jobs=-1))]),
    'extra_trees_baseline': Pipeline([('prep', make_preprocessor()), ('model', ExtraTreesClassifier(n_estimators=450, min_samples_leaf=2, class_weight='balanced', random_state=42, n_jobs=-1))]),
    'gradient_boosting_baseline': Pipeline([('prep', make_preprocessor()), ('model', GradientBoostingClassifier(n_estimators=220, learning_rate=0.06, max_depth=3, random_state=42))]),
    'svm_rbf_baseline': Pipeline([('prep', make_preprocessor()), ('model', SVC(C=7.5, gamma='scale', class_weight='balanced', probability=True, random_state=42))]),
    'mlp_deep_baseline': Pipeline([('prep', make_preprocessor()), ('model', MLPClassifier(hidden_layer_sizes=(96,48), alpha=0.0008, learning_rate_init=0.001, max_iter=450, early_stopping=True, n_iter_no_change=25, random_state=42))]),
}
results = [evaluate_model(name, model) for name, model in baselines.items()]
leaderboard = pd.DataFrame([{k:v for k,v in r.items() if k != 'estimator'} for r in results]).sort_values('test_f1_macro', ascending=False)
display(leaderboard.style.format({'cv_accuracy':'{:.3f}', 'cv_f1_macro':'{:.3f}', 'test_accuracy':'{:.3f}', 'test_f1_macro':'{:.3f}'}))

## 6. Repeated fine tuning with visible trial values

In [ ]:
cv4 = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)
searches = [
    ('random_forest_tuned', Pipeline([('prep', make_preprocessor()), ('model', RandomForestClassifier(random_state=42, n_jobs=-1))]), {'model__n_estimators':[260,420,620], 'model__max_depth':[None,12,18], 'model__min_samples_leaf':[1,2,4], 'model__class_weight':['balanced','balanced_subsample']}),
    ('extra_trees_tuned', Pipeline([('prep', make_preprocessor()), ('model', ExtraTreesClassifier(random_state=42, n_jobs=-1))]), {'model__n_estimators':[320,520,720], 'model__max_depth':[None,14,22], 'model__min_samples_leaf':[1,2,3], 'model__class_weight':['balanced','balanced_subsample']}),
    ('svm_rbf_tuned', Pipeline([('prep', make_preprocessor()), ('model', SVC(probability=True, random_state=42))]), {'model__C':[2.5,5.0,8.0,12.0], 'model__gamma':['scale',0.04,0.08], 'model__class_weight':['balanced']}),
    ('mlp_deep_tuned', Pipeline([('prep', make_preprocessor()), ('model', MLPClassifier(early_stopping=True, max_iter=550, random_state=42))]), {'model__hidden_layer_sizes':[(64,32),(96,48),(128,64,24)], 'model__alpha':[0.0003,0.0008,0.0015], 'model__learning_rate_init':[0.0007,0.001,0.0015]}),
]

for name, estimator, grid in searches:
    print('\nTuning', name)
    search = GridSearchCV(estimator, grid, scoring='f1_macro', cv=cv4, n_jobs=-1)
    search.fit(X_train, y_train)
    print('best score:', round(search.best_score_, 3))
    print('best params:', search.best_params_)
    res = evaluate_model(name, search.best_estimator_)
    res['best_params'] = search.best_params_
    results.append(res)

leaderboard = pd.DataFrame([{k:v for k,v in r.items() if k != 'estimator'} for r in results]).sort_values(['test_f1_macro','cv_f1_macro'], ascending=False)
display(leaderboard[['model','cv_accuracy','cv_f1_macro','test_accuracy','test_f1_macro']].style.format('{:.3f}', subset=['cv_accuracy','cv_f1_macro','test_accuracy','test_f1_macro']))

## 7. Optional Keras deep learning trial

In [ ]:
try:
    import tensorflow as tf
    from sklearn.preprocessing import LabelEncoder
    from sklearn.utils.class_weight import compute_class_weight

    prep = make_preprocessor()
    Xtr = prep.fit_transform(X_train, y_train)
    Xte = prep.transform(X_test)
    le = LabelEncoder()
    ytr = le.fit_transform(y_train)
    yte = le.transform(y_test)
    weights = compute_class_weight(class_weight='balanced', classes=np.unique(ytr), y=ytr)
    tf.keras.utils.set_random_seed(42)
    keras_model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(Xtr.shape[1],)),
        tf.keras.layers.Dense(96, activation='relu'),
        tf.keras.layers.Dropout(0.18),
        tf.keras.layers.Dense(48, activation='relu'),
        tf.keras.layers.Dropout(0.12),
        tf.keras.layers.Dense(len(le.classes_), activation='softmax')
    ])
    keras_model.compile(optimizer=tf.keras.optimizers.Adam(0.001), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    keras_model.fit(Xtr, ytr, validation_split=0.15, epochs=80, batch_size=64, class_weight=dict(enumerate(weights)), verbose=0, callbacks=[tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)])
    pred = le.inverse_transform(keras_model.predict(Xte, verbose=0).argmax(axis=1))
    print('Keras dense test accuracy:', round(accuracy_score(y_test, pred), 3))
    print('Keras dense test macro F1:', round(f1_score(y_test, pred, average='macro'), 3))
except Exception as exc:
    print('TensorFlow trial skipped:', exc)

## 8. Final evaluation and save the backend model

In [ ]:
best_row = leaderboard.iloc[0]
best = next(r for r in results if r['model'] == best_row['model'])
best_model = best['estimator']
pred = best_model.predict(X_test)
print('Selected:', best['model'])
print(classification_report(y_test, pred))

cm = confusion_matrix(y_test, pred, labels=sorted(y.unique()))
plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=sorted(y.unique()), yticklabels=sorted(y.unique()))
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Final blood model confusion matrix')
plt.tight_layout()
plt.show()

bundle = {
    'model': best_model,
    'features': FEATURES,
    'raw_feature_map': COL_MAP,
    'classes': sorted(y.unique().tolist()),
    'trained_on': DATA_PATH.name,
    'n_samples_train': len(X_train),
    'n_samples_test': len(X_test),
    'leaderboard': leaderboard.to_dict(orient='records'),
    'selected_model': best['model'],
    'test_accuracy': float(best['test_accuracy']),
    'test_f1_macro': float(best['test_f1_macro']),
    'notes': 'CBC-only deployment model trained from the real Excel dataset.',
}
joblib.dump(bundle, MODELS_DIR / 'blood_model.pkl')
print('Saved:', MODELS_DIR / 'blood_model.pkl')